# Weekly lot sizing at the plants

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/lot_sizing.ipynb)

A company assembles electric bicycles in three plants and plans, with M4, how many bikes each plant assembles, ships to the distribution network and holds each month from January to June. M4 is the master plan of the company. It counts bikes, not models, and gives each plant a capacity in bikes per month, as if every month had 20 working days and a bike always took the average assembly time. The plants assemble four models, City, Trekking, Mountain and Cargo, on one assembly line each. A change of model on the line is a setup: it takes time, in which nothing is assembled, and it has a cost.

This notebook takes from the master plan two figures for each plant and month: the bikes that the plant ships to the network and the stock it keeps at the end of the month. With them it plans, for each plant and week, which models the line assembles and how many bikes of each, with the assembly and setup times of the plant, its calendar and the orders that the retailers place for each model. The weekly plan shows whether the capacity that the master plan gave each plant is enough to assemble what it assigned. The hours that the line needs above its available capacity are the capacity overload, which the plant covers with overtime; the hours it leaves unused are the spare capacity.

Contents of the notebook:

1. Orders of the retailers.
2. Master plan: M4 with those orders.
3. Shipments and least stock of a plant.
4. Weekly lot plan of a plant.
5. Validation with two limit cases.
6. Hours of the line of the three plants, by month.
7. Weekly plans with one datum changed.
8. Weekly plans under the assumptions of the master plan.

The cells run in order, with ▶ or *Shift + Enter*. Cells 1 to 8 have a form (drop-down lists, sliders and tick boxes): after changing a value, run the cell again. Outside Colab the form is a set of lines such as `plant = "Plant 1"`: edit the value and run the cell.

The table gives the symbols of the weekly plan of one plant. The plan of each month covers its weeks: those with most of their days from Monday to Friday in that month (weeks 1–5, 6–9, 10–13, 14–17, 18–22 and 23–26).

| Symbol | Meaning | Term | Unit |
|---|---|---|---|
| *Sets* | | | |
| $J$, $j$ | The four bike models | model | — |
| $W$, $w$ | The weeks of the plan of one month; $\bar w$ is the last one, the week of the end of the month | — | — |
| *Data* | | | |
| $a_j$ | Time that one bike of model $j$ takes on the line | assembly time | h/bike |
| $a^{\text{set}}_j$ | Time that a change of the line to model $j$ takes, in which nothing is assembled | setup time | h/setup |
| $c^{\text{set}}_j$ | Cost of a setup of model $j$ | setup cost | €/setup |
| $c^h_j$ | Cost of holding one bike of model $j$ for one week at the plant | holding cost | €/(bike·week) |
| $K_w$ | Hours of the line in week $w$: its working days times 8 h | available capacity | h |
| $K^h$ | Bikes that the plant can hold | storage capacity | bikes |
| $x_{jw}$ | Bikes of model $j$ that the plant ships to the network in week $w$ | shipments | bikes |
| $h^{\min}_j$ | Stock of model $j$ that the plant must hold at the end of the month, from the master plan | least stock | bikes |
| *Variables* | | | |
| $p_{jw}$ | Bikes of model $j$ assembled in week $w$; the bikes of a model assembled in one week are one lot | — | bikes (integer) |
| $h_{jw}$ | Bikes of model $j$ in stock at the plant at the end of week $w$ | stock | bikes (integer) |
| $\delta_{jw}$ | 1 if model $j$ is set up and assembled in week $w$, 0 otherwise | — | — (0 or 1) |
| $o_w$ | Hours that the line needs in week $w$ above $K_w$, covered with overtime | capacity overload | h |

The [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/lot_sizing.md) explains all the data and results. The places and distances of the network are real. Original case by J. J. Ramos (UAB, 2026).

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import lot_sizing as ls

## 1. Orders of the retailers

Each week, each of the six retailers orders a number of bikes of each model, whole bikes. The cell gives the orders of the 26 weeks by model, all retailers together, and the product mix of the six months: the share of each model in all the orders, with which M4 obtained the capacity of each plant. The first figure gives the orders of each retailer in each month, the four models together: the data with which M4 is solved in §2. The second figure gives the orders of each retailer in each week, stacked by model, with the same scale for the six retailers and a dotted line at the end of each month.

`show_tables`: instead of the two figures, the orders of each retailer in each month in a table, and the orders of the retailer of the form in each week, by model, with the total of each week and of the 26 weeks. `retailer`: the retailer of the weekly table. From January to May the orders of every retailer grow: in the weekly orders of a retailer, do the four models grow in the same proportion?

In [ ]:
# @title Orders of the retailers { display-mode: "form" }
retailer = "Retailer 1 (Lleida)"  # @param ["Retailer 1 (Lleida)", "Retailer 2 (Sabadell)", "Retailer 3 (Alicante)", "Retailer 4 (Murcia)", "Retailer 5 (Córdoba)", "Retailer 6 (Granada)"]
show_tables = False  # @param {type:"boolean"}
ls.show_orders(retailer, show_table=show_tables)

## 2. Master plan: M4 with the orders of the retailers

M4 is the master plan of the running example of the topic. It is solved with the orders of each retailer in each month of §1, with no other change to its model or its data. The cell first draws the network of M4 with its data: under the label of each plant, its unit assembly cost, capacity, holding cost and storage capacity; under that of each warehouse, its holding cost and storage capacity, and the node capacity of regional warehouse 2, the bikes it can receive in a month; under that of each retailer, its backorder share, the share of the orders not received in their month that is backordered, the rest being lost sales; and on each arc, its transport cost and arc capacity. The line above the figure gives the backorder cost and the lost-sale cost. Then it gives the cost of the plan and a figure of what the plan sets for each plant: above, the bikes it has to assemble and ship to the network in each month, with its capacity as a dashed line; below, the stock it has to hold at the end of each month, stacked by model, which is the least stock of each model. The least stock is the stock that M4 keeps at the plant, split among the models as the seasonal stock: with the product mix of the orders of the later months in which that stock is used. The last line gives the stock of the network, plants and warehouses, at the end of each month. `show_tables`: instead of the two figures, the data of the warehouses, the arcs and the retailers, and the plan of each plant, in tables. The tick box changes only how the cell shows them.

In January plant 1 assembles 250 bikes and ships 230; in May it assembles 250 and ships 290. Where do the 40 bikes of May come from, and why does M4 assemble them months before?

In [ ]:
# @title Master plan { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
ls.show_master_plan(show_tables=show_tables)

## 3. Shipments and least stock of a plant

The master plan gives each plant two figures for each month, all models together: its shipments to the network and its stock at the end of the month (§2). The weekly plan needs them by model and, the shipments, by week. Three terms are used:

- *Seasonal stock*: the stock of the whole network, plants and warehouses, at the end of a month. It is built from January to March and used in April and May, the months of the peak of the orders.
- *Product mix of the shipments*: the share of each model in the shipments of a month, the same at every plant: the product mix of what the network delivers and stores in that month.
- *Least stock*: the stock at the end of the month of a plant, split among the models with the shares of the seasonal stock, in whole bikes. The weekly plan requires at least that stock of each model at the end of the last week of the month: weeks 5, 9, 13, 17, 22 and 26.

The shipments of a month are spread evenly over its days from Monday to Friday, which gives the shipments of each week by model.

The first figure gives, in %, the product mix of the shipments of each month and the split of the seasonal stock at the end of each month, with the stock of the network above each bar; it is the same for the three plants. The second figure gives, for the plant of the form, its shipments of each week, stacked by model, and, below, its least stock at the end of each month, stacked by model, in the week in which it is required.

`plant`: the plant shown in the second figure. `show_tables`: instead of the two figures, the same data in tables.

Compare, in the first figure, the product mix of the shipments of January with the split of the seasonal stock at the end of January. Which models have a larger share in the stock than in the shipments, and why?

In [ ]:
# @title Shipments and least stock { display-mode: "form" }
plant = "Plant 1"  # @param ["Plant 1", "Plant 2", "Plant 3"]
show_tables = False  # @param {type:"boolean"}
ls.show_shipments(plant, show_tables=show_tables)

## 4. Weekly lot plan of a plant

The plant makes one plan for each month, with the weeks of that month, one plan after the other: each plan starts with the stock that the previous one left. Each week starts with a setup, and each further lot of the week is a change of model. The plan of one month is

$$
\begin{aligned}
& h_{j,w-1} + p_{jw} = x_{jw} + h_{jw} && \forall j \in J,\ w \in W && \text{(balance between periods)} \\
& \sum_{j \in J} a_j\, p_{jw} + \sum_{j \in J} a^{\text{set}}_j\, \delta_{jw} \le K_w + o_w && \forall w \in W && \text{(capacity of the line, relaxed)} \\
& p_{jw} \le (x_{jw} + K^h)\, \delta_{jw} && \forall j \in J,\ w \in W && \text{(a model is assembled only if it is set up)} \\
& \sum_{j \in J} h_{jw} \le K^h && \forall w \in W && \text{(storage capacity)} \\
& h_{j\bar w} \ge h^{\min}_j && \forall j \in J && \text{(least stock at the end of the month)} \\
& p_{jw},\ h_{jw} \in \mathbb{Z}_{\ge 0},\ \delta_{jw} \in \{0; 1\},\ o_w \ge 0
\end{aligned}
$$

solved in two stages: first the least capacity overload, $\min \sum_{w} o_w$; then, among the plans with that capacity overload, the least setup and holding cost, $\min \sum_{j, w} \left( c^{\text{set}}_j\, \delta_{jw} + c^h_j\, h_{jw} \right)$. Each plan is solved to its exact optimum.

The cell gives the setup and holding cost of the six months and three figures with the same weeks. The first gives the lots: the bikes of each model assembled in each week, one lot for each model with bikes. The second gives the stock at the end of each week, by model, with the storage capacity as a dashed line and the least stock of the month, all models together, as a thick line in the week in which it is required. The third draws the hours of assembly and of setups of each week against the available capacity and, below, in red, the capacity overload of each week: the hours the line needs above the available capacity. It has its own scale, so that an overload of a few minutes can be seen, and its hours are written above each bar.

`plant`: the plant whose weekly plan is shown.

`check_other_plans`: also looks for another optimal plan in each month, with the same capacity overload and the same setup and holding cost but different lots, because some bikes are assembled in another week. The cell names the months that have one; in those months, the plan shown is one of several optimal plans.

`show_tables`: the plan in one table, instead of the three figures.

Plant 3 assembles each model in lots that cover more weeks of shipments than those of plants 1 and 2. Why?

In [ ]:
# @title Weekly lot plan { display-mode: "form" }
plant = "Plant 1"  # @param ["Plant 1", "Plant 2", "Plant 3"]
check_other_plans = False  # @param {type:"boolean"}
show_tables = False  # @param {type:"boolean"}
ls.show_weekly_plan(plant, check=check_other_plans, show_tables=show_tables);

## 5. Validation with two limit cases

The model of §4 always has a solution, because the capacity of the line is relaxed with the capacity overload: that a solution exists says nothing about whether the model is right. The cell checks it with two limit cases, in which the optimal plan is known without solving the model. In both, every week has an available capacity of 1.000 h, so that the capacity of the line never binds.

In case A, holding a bike costs 10.000 €/(bike·week). No bike is worth assembling before the week in which it is shipped, so the expected plan assembles each week, model by model, what the plant ships that week less the stock it holds, and in the last week of the month also the least stock.

In case B, a setup takes 100 h and costs 4.020 €, and the storage capacity has no limit. Holding a bike for the rest of the month costs at most 9,04 €, four weeks at the highest holding cost, 2,26 €/(bike·week), so no setup is worth making twice in a month. The expected plan assembles one lot of each model in each month, in the first week whose shipments the stock does not cover.

The cell solves the model for the three plants and says, for each, whether its plan is the expected plan. The first figure gives the lots of each model at the plant of the form: the bars are the plan of the model and the black marks the expected plan. The second gives the plan of the model as in §4: its lots and its stock.

A model that fails a limit case has a constraint or an objective that is wrong. A model that passes both can still represent the plant badly: whether its assembly and setup times are the real ones, or whether leaving out the cost and the limit of overtime is reasonable, can only be checked against the real plans and hours of the plant.

`limit_case`: case A, expensive holding, or case B, expensive setups.

`plant`: the plant whose plans are drawn.

`show_tables`: the lots of the two plans and the plan of the model in tables, instead of the figures.

Which constraints of the model, or which part of its objective, does each limit case check?

In [ ]:
# @title Validation with two limit cases { display-mode: "form" }
limit_case = "A · expensive holding"  # @param ["A · expensive holding", "B · expensive setups"]
plant = "Plant 1"  # @param ["Plant 1", "Plant 2", "Plant 3"]
show_tables = False  # @param {type:"boolean"}
ls.show_limit_case(plant, limit_case, show_tables=show_tables);

## 6. Hours of the line of the three plants, by month

The cell adds up the weekly plans of §4 by month, for the three plants: the available capacity, the hours of assembly and of setups, the capacity overload and the spare capacity, the hours that the line leaves unused. The first figure gives these hours for each plant and month, with the available capacity as a black line and, in red, the capacity overload and its share of the available capacity of the month. A month can have a capacity overload in some weeks and spare capacity in others. The second figure draws the hours of each week of the three plants, each with its capacity overload below, as in §4.

`show_tables`: the hours of each plant and month in one table, instead of the figures.

M4 gave plants 1 and 2 a capacity of 250 and 200 bikes a month, and both assemble at that capacity every month. In which months is the capacity overload of plant 2 largest, and in which weeks of those months does it fall?

In [ ]:
# @title Hours of the line by month { display-mode: "form" }
show_tables = False  # @param {type:"boolean"}
ls.show_line_hours(show_tables=show_tables);

## 7. Weekly plans with one datum changed

The cell solves again the plans of the six months of one plant with some of its data changed, and gives, in a figure, the capacity overload and the spare capacity of each month next to those with the data of the case, and the weekly plan in the figures of §4, with the hours of each week with the data of the case as a dashed line. The shipments and the least stock do not change: the master plan stays as it is.

- `no_holidays`: every week has five working days.
- `extra_holiday_week`: one more holiday, on the Wednesday of that week; 0, none.
- `model`, `setup_time` and `assembly_time`: the setup time (h/setup) and the assembly time (h/bike) of one model at the plant; 0 keeps the time of the case. The setup cost follows the setup time: 40 €/h of the changeover crew plus 20 € for the check of the first bike.
- `storage_capacity`: the bikes that the plant can hold; 0 keeps the capacity of the case. Below the least stock of a month the plan of that month has no solution, and the cell says so.
- `show_tables`: the hours of each month, next to those with the data of the case, and the weekly plan in tables, instead of the figures.

Change one datum at a time. At plant 2, give one model at a time the setup time of plant 1 (City and Trekking 2 h, Mountain 3 h, Cargo 4 h): which model lowers the capacity overload of the six months most? At plant 1, what does a storage capacity of 45 bikes change in March and April?

In [ ]:
# @title Weekly plans with one datum changed { display-mode: "form" }
plant = "Plant 2"  # @param ["Plant 1", "Plant 2", "Plant 3"]
no_holidays = False  # @param {type:"boolean"}
extra_holiday_week = 0  # @param {type:"slider", min:0, max:26, step:1}
model = "Cargo"  # @param ["City", "Trekking", "Mountain", "Cargo"]
setup_time = 0  # @param {type:"slider", min:0, max:8, step:0.25}
assembly_time = 0  # @param {type:"slider", min:0, max:2, step:0.025}
storage_capacity = 0  # @param {type:"slider", min:0, max:80, step:1}
show_tables = False  # @param {type:"boolean"}
ls.change_test(plant, no_holidays=no_holidays, extra_holiday_week=extra_holiday_week, model=model,
               setup_time=setup_time, assembly_time=assembly_time, storage=storage_capacity, show_tables=show_tables);

## 8. Weekly plans under the assumptions of the master plan

M4 gave each plant a capacity in bikes per month that rests on three assumptions: every month has 20 working days; a bike takes the same assembly time in every month, the average of the four models weighted by the product mix of the orders of the six months; and the 23 h a month that the assembly does not use are left for the setups, the same in every month and at every plant. Each run solves the weekly plans of the six months again with data in which one of these assumptions holds:

- `No holidays`: every week has five working days.
- `Half the setup times`: the four models at every plant.
- `Setup times of plant 1`: at plants 2 and 3.
- `Six-month product mix in every week`: the bikes that each plant ships in each week, all models together, are split among the four models with the product mix of the six months, and so is the least stock at the end of each month, except at the end of June.

In the runs of the setups, the setup cost follows the setup time. `As planned` are the plans of §4. The figure gives the capacity overload (above) and the spare capacity (below) of each plant and month in each run. The line below it names the months with another optimal plan: a plan with the same capacity overload and the same cost that assembles part of a lot in another week.

`plant` and `run`: with a plant, the cell also gives its weekly plan in that run in the figures of §4, with the hours of the plan `As planned` as a dashed line; with `none`, only the runs.

`show_tables`: the capacity overload and the spare capacity of each run, and the weekly plan of the plant, in tables, instead of the figures.

Which run lowers the capacity overload of plant 2 most, and in which months does it remain? With the same product mix in every week, does the capacity overload fall?

In [ ]:
# @title Weekly plans under the assumptions of the master plan { display-mode: "form" }
plant = "none"  # @param ["none", "Plant 1", "Plant 2", "Plant 3"]
run = "As planned"  # @param ["As planned", "No holidays", "Half the setup times", "Setup times of plant 1", "Six-month product mix in every week"]
show_tables = False  # @param {type:"boolean"}
ls.show_runs(plant, run, show_tables=show_tables);

The script `lot_sizing_solver.py`, in the folder of the case, prints every table of the [document of the example](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/lot_sizing.md).